## 6. Usage Notes & Next Steps

### How This Notebook Works

This notebook **automatically discovers and analyzes all model metrics** in your project:

1. **Scans Artifacts Directory**: Looks for `metrics.json` files in known locations:
   - `artifacts/baseline/metrics.json`
   - `artifacts/transformer/metrics.json`
   - Any subdirectories with model evaluation results

2. **Generates Visualizations**:
   - Overall performance comparison (accuracy, F1-score)
   - Precision, recall, and F1 breakdowns
   - Heatmap showing normalized metric scores
   - Individual model performance cards
   - Radar charts for multi-dimensional analysis

3. **Exports Results**: 
   - Metrics summary as CSV for further analysis
   - PNG images of all visualizations
   - Detailed ranking and recommendations

### To Run This Notebook

```python
# Simply run all cells in order. The notebook will:
# 1. Load any existing metrics from artifacts/
# 2. Create comparison dataframes
# 3. Generate visualizations
# 4. Print summary statistics
```

### To Add More Models

Train additional models and ensure their `metrics.json` file is saved to:
- `artifacts/[model_name]/metrics.json` (for known models)
- `artifacts/[test|prepared]-*/model/metrics.json` (for experimental runs)

Re-run this notebook and it will automatically include the new models in all comparisons.

### Recommended Next Steps

- **If one model clearly dominates**: Consider deployment or further fine-tuning
- **If models are similar**: Analyze error cases to understand model strengths/weaknesses
- **If performance is low**: Review data quality, augmentation, or try different architectures
- **For production**: Test on holdout data and monitor performance drift over time

In [ ]:
# Create radar charts for multi-dimensional comparison
import math

# Get numeric columns for radar chart
numeric_cols = [col for col in metrics_df.columns if col != "Model" and metrics_df[col].dtype in ['float64', 'float32']]

if len(numeric_cols) >= 3:  # Need at least 3 dimensions for meaningful radar chart
    num_models = len(metrics_df)
    num_vars = len(numeric_cols)
    
    # Create subplot for each model
    num_cols_plot = min(3, num_models)
    num_rows_plot = (num_models + num_cols_plot - 1) // num_cols_plot
    
    fig, axes = plt.subplots(num_rows_plot, num_cols_plot, 
                            figsize=(5*num_cols_plot, 5*num_rows_plot),
                            subplot_kw=dict(projection='polar'))
    
    axes = axes.flatten() if num_models > 1 else [axes]
    
    # Calculate angles for each metric
    angles = [n / float(num_vars) * 2 * math.pi for n in range(num_vars)]
    angles += angles[:1]  # Complete the circle
    
    colors_list = plt.cm.Set3(np.linspace(0, 1, num_models))
    
    for idx, (_, row) in enumerate(metrics_df.iterrows()):
        ax = axes[idx]
        
        # Get values for this model
        values = [row[col] for col in numeric_cols]
        values += values[:1]  # Complete the circle
        
        # Plot
        ax.plot(angles, values, 'o-', linewidth=2, label=row["Model"], color=colors_list[idx])
        ax.fill(angles, values, alpha=0.25, color=colors_list[idx])
        
        # Set labels
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([col.replace('_', '\n').title() for col in numeric_cols], size=9)
        ax.set_ylim(0, 1)
        ax.set_title(row["Model"], size=12, fontweight='bold', pad=20)
        ax.grid(True)
    
    # Hide unused subplots
    for idx in range(num_models, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig("radar_chart_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Radar chart saved as 'radar_chart_comparison.png'")

: 

### 5.1 Radar Chart (Multi-dimensional Comparison)

In [ ]:
# Generate comprehensive summary report
print("\n" + "="*100)
print("MODEL PERFORMANCE SUMMARY REPORT")
print("="*100)

# Best models by each metric
for col in metrics_df.columns[1:]:
    if metrics_df[col].dtype in ['float64', 'float32']:
        best_idx = metrics_df[col].idxmax()
        best_model = metrics_df.loc[best_idx, "Model"]
        best_score = metrics_df.loc[best_idx, col]
        print(f"\n🏆 Best {col.upper().replace('_', ' ')}: {best_model}")
        print(f"   Score: {best_score:.4f}")

# Overall ranking
print("\n" + "-"*100)
print("OVERALL MODEL RANKING")
print("-"*100)

if "f1_macro" in metrics_df.columns:
    ranking = metrics_df[["Model", "f1_macro", "accuracy"]].copy()
    ranking["score"] = ranking["f1_macro"] * 0.6 + ranking["accuracy"] * 0.4
    ranking = ranking.sort_values("score", ascending=False).reset_index(drop=True)
    
    for idx, (_, row) in enumerate(ranking.iterrows(), 1):
        print(f"{idx}. {row['Model']}")
        print(f"   F1-Macro: {row['f1_macro']:.4f} | Accuracy: {row['accuracy']:.4f} | Combined Score: {row['score']:.4f}")

# Key insights
print("\n" + "-"*100)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("-"*100)

if len(metrics_df) > 1:
    print(f"\n✓ Total Models Evaluated: {len(metrics_df)}")
    
    # Accuracy range
    if "accuracy" in metrics_df.columns:
        acc_min = metrics_df["accuracy"].min()
        acc_max = metrics_df["accuracy"].max()
        acc_range = acc_max - acc_min
        print(f"✓ Accuracy Range: {acc_min:.4f} - {acc_max:.4f} (Variation: {acc_range:.4f})")
    
    # F1 range
    if "f1_macro" in metrics_df.columns:
        f1_min = metrics_df["f1_macro"].min()
        f1_max = metrics_df["f1_macro"].max()
        f1_range = f1_max - f1_min
        print(f"✓ F1-Macro Range: {f1_min:.4f} - {f1_max:.4f} (Variation: {f1_range:.4f})")
    
    # Precision-Recall balance
    if all(m in metrics_df.columns for m in ["precision_macro", "recall_macro"]):
        for idx, (_, row) in enumerate(metrics_df.iterrows()):
            prec = row["precision_macro"]
            rec = row["recall_macro"]
            balance = 1 - abs(prec - rec)
            print(f"\n✓ {row['Model']}:")
            print(f"   Precision-Recall Balance: {balance:.4f}", end="")
            if prec > rec:
                print(" (⚠️ More precise, might miss some positives)")
            else:
                print(" (⚠️ More sensitive, might have false positives)")
else:
    print(f"\n✓ Single Model: {metrics_df.iloc[0]['Model']}")
    print("  Consider training additional baseline or alternative models for comparison.")

print("\n" + "="*100)

# Export metrics to CSV
export_path = "model_metrics_summary.csv"
metrics_df.to_csv(export_path, index=False)
print(f"\n📊 Metrics exported to '{export_path}'")

## 5. Summary Dashboard and Recommendations

In [ ]:
# Create individual performance cards for each model
fig, axes = plt.subplots(len(metrics_df), 1, figsize=(14, 3 * len(metrics_df)))

if len(metrics_df) == 1:
    axes = [axes]

for idx, (_, row) in enumerate(metrics_df.iterrows()):
    ax = axes[idx]
    
    # Get metrics for this model
    metrics_dict = {k: v for k, v in row.items() if k != "Model" and isinstance(v, (int, float))}
    
    if not metrics_dict:
        continue
    
    # Create bar plot
    metrics_names = list(metrics_dict.keys())
    metrics_values = list(metrics_dict.values())
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(metrics_names)))
    bars = ax.barh(metrics_names, metrics_values, color=colors, alpha=0.8)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, metrics_values)):
        ax.text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
    
    ax.set_xlim([0, 1.1])
    ax.set_xlabel('Score', fontsize=11, fontweight='bold')
    ax.set_title(f'Model: {row["Model"]}', fontsize=12, fontweight='bold', loc='left')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig("individual_model_performance.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Individual model details saved as 'individual_model_performance.png'")

### 4.3 Individual Model Performance Details

In [ ]:
# Create a heatmap of all metrics
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data for heatmap
heatmap_data = metrics_df.set_index("Model").copy()

# Normalize numeric columns to 0-1 range for better visualization
for col in heatmap_data.columns:
    if heatmap_data[col].dtype in ['float64', 'float32']:
        heatmap_data[col] = (heatmap_data[col] - heatmap_data[col].min()) / (heatmap_data[col].max() - heatmap_data[col].min() + 1e-8)

# Create heatmap
sns.heatmap(heatmap_data, annot=metrics_df.set_index("Model").round(4), 
            fmt='g', cmap='RdYlGn', cbar_kws={'label': 'Normalized Score'},
            ax=ax, linewidths=0.5, linecolor='gray', vmin=0, vmax=1)

ax.set_title('Performance Metrics Heatmap (Normalized)\nGreen=Higher, Red=Lower', 
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Models', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig("metrics_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Heatmap saved as 'metrics_heatmap.png'")

### 4.2 Heatmap of Metrics

In [ ]:
# Identify which metrics are available
available_metrics = [col for col in metrics_df.columns if col != "Model"]
primary_metrics = [m for m in available_metrics if any(x in m for x in ["accuracy", "f1", "precision", "recall"])]

# Create a comprehensive comparison visualization
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)

# Plot 1: Accuracy and F1-Macro
ax1 = fig.add_subplot(gs[0, :])
if "accuracy" in metrics_df.columns and "f1_macro" in metrics_df.columns:
    x = np.arange(len(metrics_df))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, metrics_df["accuracy"], width, label="Accuracy", alpha=0.8, color='steelblue')
    bars2 = ax1.bar(x + width/2, metrics_df["f1_macro"], width, label="F1-Macro", alpha=0.8, color='coral')
    
    ax1.set_xlabel("Model", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Score", fontsize=12, fontweight='bold')
    ax1.set_title("Accuracy vs F1-Macro Score Comparison", fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics_df["Model"], rotation=45, ha='right')
    ax1.legend(fontsize=11)
    ax1.set_ylim([0, 1.05])
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Precision, Recall, F1 (Macro)
ax2 = fig.add_subplot(gs[1, :])
if all(m in metrics_df.columns for m in ["precision_macro", "recall_macro", "f1_macro"]):
    x = np.arange(len(metrics_df))
    width = 0.25
    
    bars1 = ax2.bar(x - width, metrics_df["precision_macro"], width, label="Precision (Macro)", alpha=0.8, color='green')
    bars2 = ax2.bar(x, metrics_df["recall_macro"], width, label="Recall (Macro)", alpha=0.8, color='orange')
    bars3 = ax2.bar(x + width, metrics_df["f1_macro"], width, label="F1 (Macro)", alpha=0.8, color='red')
    
    ax2.set_xlabel("Model", fontsize=12, fontweight='bold')
    ax2.set_ylabel("Score", fontsize=12, fontweight='bold')
    ax2.set_title("Precision, Recall, F1 (Macro) Comparison", fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metrics_df["Model"], rotation=45, ha='right')
    ax2.legend(fontsize=11)
    ax2.set_ylim([0, 1.05])
    ax2.grid(axis='y', alpha=0.3)

# Plot 3: Micro metrics if available
ax3 = fig.add_subplot(gs[2, :])
if all(m in metrics_df.columns for m in ["precision_micro", "recall_micro", "f1_micro"]):
    x = np.arange(len(metrics_df))
    width = 0.25
    
    bars1 = ax3.bar(x - width, metrics_df["precision_micro"], width, label="Precision (Micro)", alpha=0.8, color='purple')
    bars2 = ax3.bar(x, metrics_df["recall_micro"], width, label="Recall (Micro)", alpha=0.8, color='brown')
    bars3 = ax3.bar(x + width, metrics_df["f1_micro"], width, label="F1 (Micro)", alpha=0.8, color='pink')
    
    ax3.set_xlabel("Model", fontsize=12, fontweight='bold')
    ax3.set_ylabel("Score", fontsize=12, fontweight='bold')
    ax3.set_title("Precision, Recall, F1 (Micro) Comparison", fontsize=14, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(metrics_df["Model"], rotation=45, ha='right')
    ax3.legend(fontsize=11)
    ax3.set_ylim([0, 1.05])
    ax3.grid(axis='y', alpha=0.3)
elif "accuracy" in metrics_df.columns:
    # If only basic metrics, show more detailed breakdown
    metric_cols = [col for col in metrics_df.columns if col != "Model"]
    if len(metric_cols) <= 4:
        x = np.arange(len(metrics_df))
        width = 0.2
        
        for i, col in enumerate(metric_cols):
            ax3.bar(x + (i - len(metric_cols)/2 + 0.5) * width, metrics_df[col], width, 
                   label=col.replace('_', ' ').title(), alpha=0.8)
        
        ax3.set_xlabel("Model", fontsize=12, fontweight='bold')
        ax3.set_ylabel("Score", fontsize=12, fontweight='bold')
        ax3.set_title("All Available Metrics Comparison", fontsize=14, fontweight='bold')
        ax3.set_xticks(x)
        ax3.set_xticklabels(metrics_df["Model"], rotation=45, ha='right')
        ax3.legend(fontsize=10)
        ax3.set_ylim([0, 1.05])
        ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("model_performance_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved as 'model_performance_comparison.png'")

## 4. Generate Performance Visualizations

### 4.1 Overall Performance Comparison

In [ ]:
# Create comprehensive metrics dataframe
def create_metrics_dataframe(all_metrics: Dict[str, Dict]) -> pd.DataFrame:
    """Convert metrics dictionary to a nicely formatted dataframe."""
    data = []
    
    for model_name, metrics in all_metrics.items():
        row = {"Model": model_name}
        row.update(metrics)
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Reorder columns for better readability
    metric_cols = [col for col in df.columns if col != "Model"]
    df = df[["Model"] + sorted(metric_cols)]
    
    return df


# Create metrics dataframe
metrics_df = create_metrics_dataframe(all_metrics)

# Round numerical values for better display
display_df = metrics_df.copy()
for col in display_df.columns[1:]:
    if display_df[col].dtype in ['float64', 'float32']:
        display_df[col] = display_df[col].round(4)

print("\n" + "="*80)
print("MODEL PERFORMANCE METRICS SUMMARY")
print("="*80)
print(display_df.to_string(index=False))
print("="*80)

# Calculate ranking by F1-macro (primary metric) and accuracy (secondary)
if "f1_macro" in metrics_df.columns:
    ranking_df = metrics_df[["Model", "f1_macro", "accuracy"]].copy()
    ranking_df = ranking_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)
    ranking_df["Rank"] = range(1, len(ranking_df) + 1)
    ranking_df = ranking_df[["Rank", "Model", "f1_macro", "accuracy"]]
    
    print("\nFIRST-STAGE RANKING (by F1-Macro Score):")
    print(ranking_df.to_string(index=False))

## 3. Calculate and Organize Metrics

In [ ]:
def load_metrics_from_directory(directory_path: str, model_name: str = None) -> Tuple[str, Dict]:
    """Load metrics.json from a directory."""
    metrics_path = Path(directory_path) / "metrics.json"
    if metrics_path.exists():
        try:
            metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
            return model_name or directory_path, metrics
        except Exception as e:
            print(f"Error loading metrics from {directory_path}: {e}")
    return None, None


def search_all_metrics(artifacts_root: str = "artifacts") -> Dict[str, Dict]:
    """Recursively search for all metrics.json files in artifacts directory."""
    artifacts_path = Path(artifacts_root)
    all_metrics = {}
    
    if not artifacts_path.exists():
        print(f"Artifacts directory not found at {artifacts_path}")
        return all_metrics
    
    # Check well-known directories first
    well_known_dirs = [
        ("Baseline", "baseline"),
        ("Transformer (DistilBERT)", "transformer"),
        ("Combined Hate Speech", "combined_hate_speech_slm"),
    ]
    
    for display_name, dir_name in well_known_dirs:
        model_path = artifacts_path / dir_name
        if model_path.exists():
            name, metrics = load_metrics_from_directory(str(model_path), display_name)
            if metrics:
                all_metrics[name] = metrics
    
    # Search test and prepared directories
    for pattern, prefix in [("test-*", "Test"), ("prepared-*", "Prepared")]:
        for subdir in artifacts_path.glob(pattern):
            if subdir.is_dir():
                metrics_path = subdir / "model" / "metrics.json"
                if metrics_path.exists():
                    try:
                        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
                        # Use directory name with prefix
                        model_key = f"{prefix}: {subdir.name}"
                        all_metrics[model_key] = metrics
                    except Exception as e:
                        pass
    
    return all_metrics


# Load all metrics
print("Loading all available model metrics...")
all_metrics = search_all_metrics("artifacts")

if all_metrics:
    print(f"\n✓ Found metrics for {len(all_metrics)} model(s):")
    for model_name in all_metrics.keys():
        print(f"  - {model_name}")
else:
    print("⚠ No metrics found in artifacts directory. Using sample data for demonstration...")
    # Create sample data for demonstration
    all_metrics = {
        "Transformer (DistilBERT)": {
            "accuracy": 0.9504,
            "precision_macro": 0.8481701461749119,
            "recall_macro": 0.8550393500888551,
            "f1_macro": 0.8515610036391497
        }
    }

## 2. Load Model Results

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

## 1. Import Required Libraries

# Model Performance Analysis and Comparison

This notebook aggregates and visualizes the performance metrics of all trained models for the Toxic Comment Detection task.